# s06 — The `revert` Tool

**What this teaches:** how Omnis keeps a per-path snapshot of every `write` (and other file-mutating tools), so the model can undo its own mistakes by calling `revert <path>`. This is the safety net that makes agentic file-editing tolerable.

**Concept anchor:** an LLM will eventually clobber the wrong file. A reliable agent isn't one that never errs — it's one that *can be told to undo*. `revert` makes that contract explicit and cheap.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [docs/providers.md](../../docs/providers.md)).
- Write access to the notebook's working directory — the agent will create, modify, and revert `demo.txt`.

## 1. Imports

Same imports as [s05_tools](../s05_tools/s05_tools.ipynb): `agentkit`, `stream`, and `fstools` (where `revert` lives).

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    fstools "github.com/blouargant/omnis/core/tools"
)

## 2. Helper

Panic-based `must`; keeps the kernel running through errors.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Construct the model and agent

We turn on the full tool set (`fstools.New()`) — `revert` is included by default. The `Instruction` is the meaningful change here: it tells the model exactly when to use `revert`, which improves reliability on weaker models.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s06_revert",
    Description: "Demonstrates the revert tool.",
    Instruction: "When asked, write to a file, modify it, then call revert and confirm contents.",
    Model:       llm,
    Tools:       fstools.New(),
})
must(err)

## 4. Build the runner and run

The prompt walks the agent through a `write → overwrite → revert → read` cycle. After `revert`, the contents of `demo.txt` should be `first` again — the previous snapshot.

In [ ]:
r, err := agentkit.Runner("s06", a)
must(err)

prompt := "Write demo.txt with 'first', then overwrite with 'second', then revert. Read the result."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- Two `write` calls, one `revert` call, then a final `read` reporting `first` — the *pre-overwrite* snapshot.
- `revert` operates per-path: each tracked file keeps its own undo stack. Multiple writes followed by one `revert` step back one level only.
- For the broader file-system tool set, see [s05_tools](../s05_tools/s05_tools.ipynb).

## Try it yourself

1. Modify the prompt to perform three sequential writes, then two reverts — observe whether the agent ends up at the first or second snapshot.
2. Ask the agent to revert a file it never wrote to — expect a tool error reported back to the model rather than silent success.